# CARE Scenario Analysis — ATE Calculator

Computes Average Treatment Effects from the estimated ordered probit models, with **every scenario
dimension selectable**: event, activity, base severity level, comparison severity level, and response
category. Panel 2 (population segments) is included too.

This notebook does **not** re-estimate anything — it reads the fitted coefficients from
`model_coefficients_all_events.csv` (all 5 events, final models only) and the event data files, so it
runs in seconds. Re-estimation lives in `Data_Preparation_and_Modeling.ipynb`.

**Inputs expected in the working directory**
- `model_coefficients_all_events.csv`
- `event_data/heat_data.csv`, `cold_data.csv`, `powerout_data.csv`, `earthquake_data.csv`, `flooding_data.csv`

## What an ATE is here

For a chosen manipulated variable (severity level, or a demographic):

1. Take the estimation sample of that activity's final model.
2. **Base scenario** — force the variable to its base value for *every* person, leave everything else at
   observed values, compute each person's `P(category)`.
3. **Comparison scenario** — force it to the comparison value, recompute.
4. Average over people → `P_base`, `P_comp`.
5. `Absolute ATE = P_comp − P_base` (percentage points); `Percent ATE = (P_comp − P_base) / P_base`.

Ordered probit: `P(Y=1) = Φ(τ₁ − z)`, `P(Y=j) = Φ(τⱼ − z) − Φ(τⱼ₋₁ − z)`, `P(Y=J) = 1 − Φ(τ_{J−1} − z)`, with `z = Xβ`.

## 1. Setup

In [1]:
import pandas as pd, numpy as np
from scipy.stats import norm
import itertools, warnings
warnings.filterwarnings('ignore')

COEF = pd.read_csv('./model_coefficients_all_events.csv')

EVENTS = {
    'heat':       dict(label='Extreme Heat',  flag='ext_heat',       file='./event_data/heat_data.csv'),
    'cold':       dict(label='Extreme Cold',  flag='ext_cold',       file='./event_data/cold_data.csv'),
    'powerout':   dict(label='Power Outage',  flag='ext_powerout',   file='./event_data/powerout_data.csv'),
    'earthquake': dict(label='Earthquakes',   flag='ext_earthquake', file='./event_data/earthquake_data.csv'),
    'flooding':   dict(label='Flooding',      flag='ext_flooding',   file='./event_data/flooding_data.csv'),
}

# activity -> dv column suffix pattern per event, and whether the model uses the working sub-sample
ACT_DV = {
 'Usual':   ('lkly_normal_business', False), 'Home':    ('stay_home',   False),
 'Car':     ('car_travel',           False), 'WFH':     ('wfh',         True),
 'WFO':     ('commute',              True),  'Dine in': ('indoor_restaurant', False),
 'Pick up': ('takeout_pickup',       False), 'Delivery':('food_delivery',    False),
}
# transit is named differently for heat than for the rest
TRANSIT_DV = {'heat':'public_transit','cold':'transit_use','powerout':'transit_use',
              'earthquake':'transit_use','flooding':'transit_use'}

ACT_LABEL = {'Usual':'Go about business as usual','Home':'Staying at home','Car':'Using a car for traveling',
 'Transit':'Taking public transit','WFH':'Working from home','WFO':'Working from the office',
 'Dine in':'Eating indoors at a restaurant','Pick up':'Picking up takeout','Delivery':'Having food delivered'}
SEV_LABEL = {1:'Not severe at all',2:'Slightly severe',3:'Moderately severe',4:'Very severe',5:'Extremely severe'}
CAT3 = {1:'Do less',2:'About the same',3:'Do more'}
CAT5 = {1:'Very unlikely',2:'Somewhat unlikely',3:'Neutral',4:'Somewhat likely',5:'Very likely'}

_data_cache = {}
def load_event(ev):
    if ev not in _data_cache:
        _data_cache[ev] = pd.read_csv(EVENTS[ev]['file'], low_memory=False)
    return _data_cache[ev]

def dv_column(ev, act):
    suffix = TRANSIT_DV[ev] if act=='Transit' else ACT_DV[act][0]
    return f'ext_{ev}_{suffix}'

def worker_only(act):
    return True if act in ('WFH','WFO') else False

def available_activities(ev):
    return sorted(COEF[COEF.event==ev].activity.unique().tolist())

print('Events:', {e: available_activities(e) for e in EVENTS})

Events: {'heat': ['Car', 'Delivery', 'Dine in', 'Home', 'Pick up', 'Transit', 'Usual', 'WFH', 'WFO'], 'cold': ['Car', 'Delivery', 'Dine in', 'Home', 'Pick up', 'Transit', 'Usual', 'WFH', 'WFO'], 'powerout': ['Car', 'Delivery', 'Dine in', 'Home', 'Pick up', 'Transit', 'Usual'], 'earthquake': ['Car', 'Home', 'Transit', 'Usual', 'WFH'], 'flooding': ['Car', 'Home', 'Transit', 'Usual', 'WFH']}


## 2. Core functions

`get_model` pulls a fitted model; `cat_probs` turns an index into category probabilities;
`ate_severity` and `ate_segment` are the two ATE engines.

In [2]:
def get_model(ev, act):
    """Return (coefs dict, thresholds list, x_vars, estimation sample, dv name)."""
    d = COEF[(COEF.event==ev) & (COEF.activity==act)]
    if d.empty:
        raise ValueError(f'No model for {act} / {ev}')
    coefs = {r.variable: r.value for r in d[d.term=='coef'].itertuples()}
    thr   = [r.value for r in d[d.term=='threshold'].sort_values('variable').itertuples()]
    x_vars = list(coefs)
    dv = dv_column(ev, act)
    sub = load_event(ev)
    if worker_only(act):
        sub = sub[sub['empsta'] < 3]
    sample = sub[x_vars + [dv]].dropna().copy()
    return coefs, thr, x_vars, sample, dv

def cat_probs(z, thr):
    """Ordered probit category probabilities for an array of indices z."""
    z = np.asarray(z, dtype=float)
    cum = np.column_stack([norm.cdf(t - z) for t in thr])
    cum = np.column_stack([np.zeros(len(z)), cum, np.ones(len(z))])
    return np.diff(cum, axis=1)

def ate_severity(ev, act, base_level, comp_level, collapse_usual=True):
    """ATE of moving everyone from base_level to comp_level of past-event severity.
    base_level / comp_level are 1..5 (1 = 'Not severe at all', the omitted reference)."""
    coefs, thr, x_vars, sample, dv = get_model(ev, act)
    sev_vars = [f'{ev}_imp_{l}' for l in [2,3,4,5]]
    nonsev = [v for v in x_vars if v not in sev_vars]
    xb = sum(sample[v].values*coefs[v] for v in nonsev) if nonsev else np.zeros(len(sample))
    add = lambda L: 0.0 if L==1 else coefs.get(f'{ev}_imp_{L}', 0.0)
    Pb = cat_probs(xb + add(base_level), thr).mean(axis=0)
    Pc = cat_probs(xb + add(comp_level), thr).mean(axis=0)
    J = len(thr)+1
    if J==5 and collapse_usual:            # 5-point likelihood -> 3 dashboard buttons
        grp = [[0,1],[2],[3,4]]
        Pb = np.array([Pb[g].sum() for g in grp]); Pc = np.array([Pc[g].sum() for g in grp])
        labs = CAT3
    else:
        labs = CAT5 if J==5 else CAT3
    out = pd.DataFrame({'Response':[labs[j+1] for j in range(len(Pb))],
                        'P_base':Pb, 'P_comp':Pc, 'ATE_abs':Pc-Pb,
                        'ATE_pct':np.where(Pb>0,(Pc-Pb)/Pb*100,np.nan)})
    out.attrs.update(event=ev, activity=act, n=len(sample),
                     base=SEV_LABEL[base_level], comp=SEV_LABEL[comp_level])
    return out

def ate_segment(ev, act, base_spec, comp_spec, collapse_usual=True):
    """ATE of moving everyone from base_spec to comp_spec, holding each person's OWN severity fixed.
    base_spec / comp_spec are dicts {variable: forced value}, e.g. {'female':0} vs {'female':1}.
    Variables that backward elimination dropped are simply absent from the model -> contribute nothing."""
    coefs, thr, x_vars, sample, dv = get_model(ev, act)
    xb_obs = sum(sample[v].values*coefs[v] for v in x_vars)
    in_model = [v for v in base_spec if v in coefs]
    if not in_model:
        J = 3 if (len(thr)+1)==3 or collapse_usual else len(thr)+1
        return pd.DataFrame({'Response':[CAT3[j+1] for j in range(3)],
                             'P_base':[np.nan]*3,'P_comp':[np.nan]*3,
                             'ATE_abs':[0.0]*3,'ATE_pct':[0.0]*3}), False
    shift_b = sum((base_spec[v]-sample[v].values)*coefs[v] for v in in_model)
    shift_c = sum((comp_spec[v]-sample[v].values)*coefs[v] for v in in_model)
    Pb = cat_probs(xb_obs+shift_b, thr).mean(axis=0)
    Pc = cat_probs(xb_obs+shift_c, thr).mean(axis=0)
    J = len(thr)+1
    if J==5 and collapse_usual:
        grp=[[0,1],[2],[3,4]]
        Pb=np.array([Pb[g].sum() for g in grp]); Pc=np.array([Pc[g].sum() for g in grp]); labs=CAT3
    else:
        labs = CAT5 if J==5 else CAT3
    out = pd.DataFrame({'Response':[labs[j+1] for j in range(len(Pb))],
                        'P_base':Pb,'P_comp':Pc,'ATE_abs':Pc-Pb,
                        'ATE_pct':np.where(Pb>0,(Pc-Pb)/Pb*100,np.nan)})
    out.attrs.update(event=ev, activity=act, n=len(sample))
    return out, True

def ate_continuous(ev, act, var, mult=0.01, collapse_usual=True):
    """ATE of an ADDITIVE shift of `mult` standard deviations in a continuous attitude score.
    A literal multiplicative '1% increase' is not usable for CR/SE/PR: they are standardized factor
    scores with mean ~ 0 and both signs, so x -> 1.01x gives an average shift of ~0 and moves
    negative-scoring people in the opposite direction. mult=0.01 is the '1% of a SD' reading."""
    coefs, thr, x_vars, sample, dv = get_model(ev, act)
    if var not in coefs:
        return pd.DataFrame({'Response':list(CAT3.values()),'P_base':[np.nan]*3,'P_comp':[np.nan]*3,
                             'ATE_abs':[0.0]*3,'ATE_pct':[0.0]*3}), False
    xb = sum(sample[v].values*coefs[v] for v in x_vars)
    delta = mult*float(sample[var].std())
    Pb = cat_probs(xb, thr).mean(axis=0); Pc = cat_probs(xb + coefs[var]*delta, thr).mean(axis=0)
    J=len(thr)+1
    if J==5 and collapse_usual:
        grp=[[0,1],[2],[3,4]]
        Pb=np.array([Pb[g].sum() for g in grp]); Pc=np.array([Pc[g].sum() for g in grp]); labs=CAT3
    else:
        labs = CAT5 if J==5 else CAT3
    out=pd.DataFrame({'Response':[labs[j+1] for j in range(len(Pb))],'P_base':Pb,'P_comp':Pc,
                      'ATE_abs':Pc-Pb,'ATE_pct':np.where(Pb>0,(Pc-Pb)/Pb*100,np.nan)})
    out.attrs.update(event=ev, activity=act, n=len(sample), shift=f'{mult} SD')
    return out, True

print('functions ready')

functions ready


## 3. Panel 1 — pick any event, base level and comparison level

Change the four values in the cell below and re-run it. **Any** ordered pair of levels works
(1→5, 3→2, 4→5, …), not just comparisons against "Not severe at all".

In [7]:
# ------------------------- CHANGE THESE -------------------------
EVENT       = 'heat'          # 'heat' | 'cold' | 'powerout' | 'earthquake' | 'flooding'
BASE_LEVEL  = 3               # 1..5   (1 = Not severe at all)
COMP_LEVEL  = 4               # 1..5
RESPONSE    = 'Do less'       # 'Do less' | 'About the same' | 'Do more'
# ----------------------------------------------------------------

rows=[]
for act in available_activities(EVENT):
    t = ate_severity(EVENT, act, BASE_LEVEL, COMP_LEVEL)
    r = t[t.Response==RESPONSE].iloc[0]
    rows.append(dict(Activity=ACT_LABEL[act], P_base=r.P_base, P_comp=r.P_comp,
                     ATE_abs=r.ATE_abs, ATE_pct=r.ATE_pct, N=t.attrs['n']))
tbl = pd.DataFrame(rows).sort_values('ATE_pct', ascending=False)

print(f"{EVENTS[EVENT]['label']}  |  {SEV_LABEL[BASE_LEVEL]}  ->  {SEV_LABEL[COMP_LEVEL]}  |  response: {RESPONSE}")
print('='*92)
print(tbl.to_string(index=False, float_format=lambda x: f'{x:8.4f}'))

Extreme Heat  |  Moderately severe  ->  Very severe  |  response: Do less
                      Activity   P_base   P_comp  ATE_abs  ATE_pct    N
    Go about business as usual   0.2607   0.3304   0.0697  26.7526 2564
Eating indoors at a restaurant   0.2303   0.2610   0.0307  13.3358 2565
       Working from the office   0.2041   0.2207   0.0166   8.1445 1271
            Picking up takeout   0.2184   0.2228   0.0044   2.0365 2564
         Taking public transit   0.2370   0.2394   0.0024   1.0234 2565
     Using a car for traveling   0.1747   0.1714  -0.0033  -1.8841 2564
         Having food delivered   0.1444   0.1055  -0.0389 -26.9479 2565
               Staying at home   0.0577   0.0390  -0.0187 -32.4031 2564
             Working from home   0.0803   0.0452  -0.0350 -43.6491 1213


### 3b. One activity, all five levels at once

Useful for spotting non-monotonic severity effects (e.g. heat/WFH peaks at *Very severe* and falls back at *Extremely severe*, because `heat_imp_5` is not significant in that model).

In [9]:
EVENT, ACTIVITY, RESPONSE = 'heat', 'WFH', 'Do more'

rows=[]
for L in [1,2,3,4,5]:
    t = ate_severity(EVENT, ACTIVITY, 1, L)
    r = t[t.Response==RESPONSE].iloc[0]
    rows.append(dict(Level=SEV_LABEL[L], P=r.P_comp,
                     ATE_abs_vs_L1=r.ATE_abs, ATE_pct_vs_L1=r.ATE_pct))
print(f'{EVENTS[EVENT]["label"]} — {ACT_LABEL[ACTIVITY]} — P({RESPONSE}) at each severity level')
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f'{x:8.4f}'))

Extreme Heat — Working from home — P(Do more) at each severity level
            Level        P  ATE_abs_vs_L1  ATE_pct_vs_L1
Not severe at all   0.2508         0.0000         0.0000
  Slightly severe   0.2460        -0.0048        -1.9069
Moderately severe   0.2394        -0.0115        -4.5746
      Very severe   0.3372         0.0863        34.4199
 Extremely severe   0.2470        -0.0039        -1.5438


### 3c. Full matrix — every base × comparison pair for one activity

The diagonal is zero by construction; the matrix is antisymmetric in absolute ATE.

In [10]:
EVENT, ACTIVITY, RESPONSE = 'heat', 'Home', 'Do more'

M = pd.DataFrame(index=[SEV_LABEL[b] for b in range(1,6)],
                 columns=[SEV_LABEL[c] for c in range(1,6)], dtype=float)
for b in range(1,6):
    for c in range(1,6):
        if b==c:
            M.iloc[b-1,c-1]=0.0; continue
        t=ate_severity(EVENT,ACTIVITY,b,c)
        M.iloc[b-1,c-1]=t[t.Response==RESPONSE].iloc[0].ATE_pct
print(f'Percent ATE (%) — {EVENTS[EVENT]["label"]}, {ACT_LABEL[ACTIVITY]}, P({RESPONSE})')
print('rows = base level, columns = comparison level')
print(M.to_string(float_format=lambda x: f'{x:+7.2f}'))

Percent ATE (%) — Extreme Heat, Staying at home, P(Do more)
rows = base level, columns = comparison level
                   Not severe at all  Slightly severe  Moderately severe  Very severe  Extremely severe
Not severe at all              +0.00           +31.79             +33.45       +55.74            +46.63
Slightly severe               -24.12            +0.00              +1.26       +18.18            +11.26
Moderately severe             -25.06            -1.24              +0.00       +16.71             +9.88
Very severe                   -35.79           -15.38             -14.32        +0.00             -5.85
Extremely severe              -31.80           -10.12              -8.99        +6.22             +0.00


## 4. Optional interactive widgets

If `ipywidgets` is installed, this gives dropdowns instead of editing code. If it isn't, the cell
prints a note and you can keep using the cells above — nothing else depends on it.

In [11]:
try:
    import ipywidgets as W
    from IPython.display import display, clear_output

    w_ev   = W.Dropdown(options=[(v['label'],k) for k,v in EVENTS.items()], value='heat', description='Event:')
    w_base = W.Dropdown(options=[(SEV_LABEL[i],i) for i in range(1,6)], value=1, description='Base:')
    w_comp = W.Dropdown(options=[(SEV_LABEL[i],i) for i in range(1,6)], value=5, description='Comparison:')
    w_resp = W.Dropdown(options=['Do less','About the same','Do more'], value='Do more', description='Response:')
    out = W.Output()

    def refresh(_=None):
        with out:
            clear_output()
            rows=[]
            for act in available_activities(w_ev.value):
                t=ate_severity(w_ev.value, act, w_base.value, w_comp.value)
                r=t[t.Response==w_resp.value].iloc[0]
                rows.append(dict(Activity=ACT_LABEL[act], P_base=r.P_base, P_comp=r.P_comp,
                                 ATE_abs=r.ATE_abs, ATE_pct=r.ATE_pct, N=t.attrs['n']))
            df_=pd.DataFrame(rows).sort_values('ATE_pct',ascending=False)
            print(f'{EVENTS[w_ev.value]["label"]} | {SEV_LABEL[w_base.value]} -> {SEV_LABEL[w_comp.value]} | {w_resp.value}')
            print('='*92)
            print(df_.to_string(index=False, float_format=lambda x: f'{x:8.4f}'))

    for w in (w_ev,w_base,w_comp,w_resp): w.observe(refresh,names='value')
    display(W.VBox([W.HBox([w_ev,w_base]), W.HBox([w_comp,w_resp]), out]))
    refresh()
except ImportError:
    print('ipywidgets not installed - use the editable cells in section 3 instead.')
    print('(pip install ipywidgets)')

## 5. Panel 2 — population segments

Severity is **not** reset here: each person keeps their own observed severity, and only the segment
variable is manipulated. A variable that backward elimination removed from that activity's model has
an ATE of exactly 0 — flagged as `in_model = False`.

In [12]:
SEGMENTS = {
 'Gender':              ('Male', {'female':0}, [('Female', {'female':1})]),
 'Age Group':           ('18-30', {'age_3150':0,'age_5165':0,'age_65p':0},
                          [('31-50',{'age_3150':1,'age_5165':0,'age_65p':0}),
                           ('51-65',{'age_3150':0,'age_5165':1,'age_65p':0}),
                           ('65+',  {'age_3150':0,'age_5165':0,'age_65p':1})]),
 'Household Income':    ('Less than $50k', {'in50':1,'in50100':0},
                          [('$50k-$100k',{'in50':0,'in50100':1}), ('$100k or higher',{'in50':0,'in50100':0})]),
 'Household Size':      ('3+ persons', {'hhsize1':0,'hhsize2':0},
                          [('1 person',{'hhsize1':1,'hhsize2':0}), ('2 persons',{'hhsize1':0,'hhsize2':1})]),
 'Child in household':  ('No child', {'child':0}, [('Has child',{'child':1})]),
 'Disability':          ('No disability', {'dis_yes':0}, [('Has disability',{'dis_yes':1})]),
 'Works outdoors':      ('No', {'work_out':0}, [('Yes',{'work_out':1})]),
 'Zero-vehicle HH':     ('Has vehicle', {'hhveh0':0}, [('Zero vehicle',{'hhveh0':1})]),
 'Stand-alone house':   ('Not stand-alone', {'sa_home':0}, [('Stand-alone house',{'sa_home':1})]),
 'Rural':               ('Not rural', {'rural':0}, [('Rural',{'rural':1})]),
 'Population Density':  ('Low', {'PopDens_medium':0,'PopDens_high':0},
                          [('Medium',{'PopDens_medium':1,'PopDens_high':0}),
                           ('High',  {'PopDens_medium':0,'PopDens_high':1})]),
 'Transit Access':      ('Low', {'TransitAccess_medium':0,'TransitAccess_high':0},
                          [('Medium',{'TransitAccess_medium':1,'TransitAccess_high':0}),
                           ('High',  {'TransitAccess_medium':0,'TransitAccess_high':1})]),
 'Walkability Index':   ('Very low', {'NatWalkInd_low':0,'NatWalkInd_high':0,'NatWalkInd_very_high':0},
                          [('Low',      {'NatWalkInd_low':1,'NatWalkInd_high':0,'NatWalkInd_very_high':0}),
                           ('High',     {'NatWalkInd_low':0,'NatWalkInd_high':1,'NatWalkInd_very_high':0}),
                           ('Very high',{'NatWalkInd_low':0,'NatWalkInd_high':0,'NatWalkInd_very_high':1})]),
}

# ------------------------- CHANGE THESE -------------------------
EVENT, ACTIVITY, RESPONSE = 'heat', 'Car', 'Do more'
# ----------------------------------------------------------------

rows=[]
for name,(base_lab, base_spec, comps) in SEGMENTS.items():
    for comp_lab, comp_spec in comps:
        t, ok = ate_segment(EVENT, ACTIVITY, base_spec, comp_spec)
        r = t[t.Response==RESPONSE].iloc[0]
        rows.append(dict(Variable=name, Base=base_lab, Comparison=comp_lab,
                         in_model=ok, P_base=r.P_base, P_comp=r.P_comp,
                         ATE_abs=r.ATE_abs, ATE_pct=r.ATE_pct))
for name,var in [('Community Resilience','CR'),('Social Engagement','SE'),('Personal Resilience','PR')]:
    for lab,mult in [('+1% of SD',0.01),('+1 SD',1.0)]:
        t, ok = ate_continuous(EVENT, ACTIVITY, var, mult=mult)
        r = t[t.Response==RESPONSE].iloc[0]
        rows.append(dict(Variable=name, Base='observed', Comparison=lab, in_model=ok,
                         P_base=r.P_base, P_comp=r.P_comp, ATE_abs=r.ATE_abs, ATE_pct=r.ATE_pct))

seg = pd.DataFrame(rows)
print(f'{EVENTS[EVENT]["label"]} — {ACT_LABEL[ACTIVITY]} — P({RESPONSE}), severity held at observed values')
print('='*104)
print(seg.to_string(index=False, float_format=lambda x: f'{x:8.4f}'))

Extreme Heat — Using a car for traveling — P(Do more), severity held at observed values
            Variable            Base        Comparison  in_model   P_base   P_comp  ATE_abs  ATE_pct
              Gender            Male            Female     False      NaN      NaN   0.0000   0.0000
           Age Group           18-30             31-50      True   0.1750   0.1750   0.0000   0.0000
           Age Group           18-30             51-65      True   0.1750   0.1528  -0.0222 -12.7007
           Age Group           18-30               65+      True   0.1750   0.1750   0.0000   0.0000
    Household Income  Less than $50k        $50k-$100k     False      NaN      NaN   0.0000   0.0000
    Household Income  Less than $50k   $100k or higher     False      NaN      NaN   0.0000   0.0000
      Household Size      3+ persons          1 person      True   0.1627   0.1889   0.0262  16.0800
      Household Size      3+ persons         2 persons      True   0.1627   0.1627   0.0000   0.0000
  C

## 6. Export

Writes every severity pair for every event and activity to one tidy CSV — 5 events × all activities ×
20 ordered level pairs × each response category. Useful as the data source behind the dashboard.

In [ ]:
rows=[]
for ev in EVENTS:
    for act in available_activities(ev):
        for b,c in itertools.permutations([1,2,3,4,5], 2):
            t = ate_severity(ev, act, b, c)
            for _,r in t.iterrows():
                rows.append(dict(event=ev, event_label=EVENTS[ev]['label'], activity=act,
                                 activity_label=ACT_LABEL[act], base_level=b, base_label=SEV_LABEL[b],
                                 comp_level=c, comp_label=SEV_LABEL[c], response=r.Response,
                                 P_base=r.P_base, P_comp=r.P_comp, ATE_abs=r.ATE_abs,
                                 ATE_pct=r.ATE_pct, n=t.attrs['n']))
allpairs = pd.DataFrame(rows)
allpairs.to_csv('./ATE_severity_all_events.csv', index=False)
print(allpairs.shape, '-> ATE_severity_all_events.csv')
print(allpairs.groupby('event_label').size())